텍스트 데이터 준비 및 개별 검색기 생성
- 사내 IT 규정 문서를 바탕으로 BM25 키워드 검색기를 1차 기본 검색기로 초기화합니다.

In [10]:
from langchain_core.documents import Document
from langchain_community.retrievers import BM25Retriever

# 실습용 사내 IT 규정 문서를 정의합니다.
raw_documents = [
    Document(page_content="보안 규정: 업무용 PC는 3개월마다 비밀번호를 변경해야 하며 특수문자를 포함해야 합니다."),
    Document(page_content="원격 근무 가이드: 재택 근무 시 반드시 사내 VPN을 연결하여 사내 인트라넷에 접속해야 합니다."),
    Document(page_content="장비 지급 규정: 신규 입사자에게는 노트북 1대와 외장 모니터 1대가 기본 지급됩니다."),
    Document(page_content="계정 관리: 사내 계정 5회 연속 로그인 실패 시 계정이 잠기며 IT 지원팀에 잠금 해제를 요청해야 합니다.")
]

# 공백 기준 단순 토크나이저 함수를 정의합니다.
def custom_tokenizer(text: str) -> list[str]:
    # 문장을 띄어쓰기 단위로 쪼개어 단어 목록을 생성합니다.
    return text.split()

# 1단계 기본 검색기(BM25)를 생성하고 후보군 추출 개수를 3개로 설정합니다.
base_retriever = BM25Retriever.from_documents(
    documents=raw_documents,
    preprocess_func=custom_tokenizer
)
# 1단계 검색에서 상위 3개의 문서를 뽑아오도록 지정합니다.
base_retriever.k = 3

 Cross-Encoder 모델을 활용한 순위 재조정(Re-ranking)
 - 오픈소스 고성능 다국어 크로스 인코더 모델을 로드하고 압축기(Compressor) 형태로 1차 검색기에 결합합니다.

In [11]:
# 허깅페이스 기반 Cross-Encoder 래퍼 클래스를 가져옵니다.
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
# 문서를 재정렬해 상위 결과를 선별하는 리랭커 모듈을 가져옵니다.
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker

# 기본 검색기와 압축기를 하나로 묶어주는 리트리버를 가져옵니다.
from langchain_classic.retrievers import ContextualCompressionRetriever

# 한국어와 영어를 지원하는 다국어 경량 Cross-Encoder 모델을 초기화합니다.
cross_encoder_model = HuggingFaceCrossEncoder(model_name="Dongjin-kr/ko-reranker")

# Cross-Encoder 기반 리랭커를 정의하고, 재정렬 후 최종 2건만 남기도록 지정합니다.
reranker = CrossEncoderReranker(
    model=cross_encoder_model,
    top_n=2
)

# 1차 검색기(base_retriever)와 2차 리랭커(reranker)를 연결합니다.
rerank_retriever = ContextualCompressionRetriever(
    base_compressor=reranker,
    base_retriever=base_retriever
)


Loading weights: 100%|██████████| 393/393 [00:00<00:00, 10509.69it/s]


In [7]:
from langchain_core.documents import Document
from langchain_community.retrievers import BM25Retriever

# 실습용 사내 IT 규정 문서를 정의합니다.
raw_documents = [
    Document(page_content="보안 규정: 업무용 PC는 3개월마다 비밀번호를 변경해야 하며 특수문자를 포함해야 합니다."),
    Document(page_content="원격 근무 가이드: 재택 근무 시 반드시 사내 VPN을 연결하여 사내 인트라넷에 접속해야 합니다."),
    Document(page_content="장비 지급 규정: 신규 입사자에게는 노트북 1대와 외장 모니터 1대가 기본 지급됩니다."),
    Document(page_content="계정 관리: 사내 계정 5회 연속 로그인 실패 시 계정이 잠기며 IT 지원팀에 잠금 해제를 요청해야 합니다.")
]

def custom_tokenizer(text):
    return text.split()

bm25_retriever = BM25Retriever.from_documents(
    documents=raw_documents,
    preprocess_func=custom_tokenizer
)

bm25_retriever.k = 3

In [8]:
# 허깅페이스 기반 Cross-Encoder 래퍼 클래스를 가져옵니다.
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
# 문서를 재정렬해 상위 결과를 선별하는 리랭커 모듈을 가져옵니다.
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
# 기본 검색기와 압축기를 하나로 묶어주는 리트리버를 가져옵니다.
from langchain_classic.retrievers import ContextualCompressionRetriever

# 한국어와 영어를 지원하는 다국어 경량 Cross-Encoder 모델을 초기화합니다.
cross_encoder_model = HuggingFaceCrossEncoder(model_name="Dongjin-kr/ko-reranker")

reranker = CrossEncoderReranker(
    model=cross_encoder_model,
    top_n=2
)

rerank_retriever = ContextualCompressionRetriever(
    base_compressor=reranker,
    base_retriever=bm25_retriever
)

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 9613.06it/s]


리랭킹 전후 검색 순위 및 점수 비교 분석
- 질문 문맥에 따라 1차 검색 결과의 순위가 리랭커를 통해 어떻게 역전되는지 확인합니다.

In [13]:
# 질문 정의: 비밀번호 실패로 인한 계정 잠금 문제를 묻는 우회적 질문
query = "사내 망 들어가려다 비밀번호 여러 번 틀렸을 때 처리 방법"

# 1차 기본 검색기(BM25)만의 결과 확인
docs_before = base_retriever.invoke(query)
print(f"=== [1차 검색 (BM25 단독)] 검색어: {query} ===")
for idx, doc in enumerate(docs_before):
    print(f"{idx + 1}순위: {doc.page_content}")

print("\n" + "=" * 60 + "\n")

# 2단계 리랭킹(BM25 + Cross-Encoder)이 적용된 최종 결과 확인
docs_after = rerank_retriever.invoke(query)
print(f"=== [2단계 리랭킹 적용] 검색어: {query} ===")
for idx, doc in enumerate(docs_after):
    print(f"{idx + 1}순위: {doc.page_content}")


=== [1차 검색 (BM25 단독)] 검색어: 사내 망 들어가려다 비밀번호 여러 번 틀렸을 때 처리 방법 ===
1순위: 계정 관리: 사내 계정 5회 연속 로그인 실패 시 계정이 잠기며 IT 지원팀에 잠금 해제를 요청해야 합니다.
2순위: 장비 지급 규정: 신규 입사자에게는 노트북 1대와 외장 모니터 1대가 기본 지급됩니다.
3순위: 원격 근무 가이드: 재택 근무 시 반드시 사내 VPN을 연결하여 사내 인트라넷에 접속해야 합니다.


=== [2단계 리랭킹 적용] 검색어: 사내 망 들어가려다 비밀번호 여러 번 틀렸을 때 처리 방법 ===
1순위: 계정 관리: 사내 계정 5회 연속 로그인 실패 시 계정이 잠기며 IT 지원팀에 잠금 해제를 요청해야 합니다.
2순위: 원격 근무 가이드: 재택 근무 시 반드시 사내 VPN을 연결하여 사내 인트라넷에 접속해야 합니다.
